In [1]:
%pip install -q pymongo python-dotenv numpy pandas 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from dotenv import load_dotenv
from pymongo import MongoClient
from datetime import datetime
import os

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["blogdb"]

collection = db["ai_news"]

collection.find_one()

{'_id': ObjectId('667d1ef6fc45eb48396a6a0c'),
 'date': datetime.datetime(2024, 6, 20, 14, 0),
 'title': "Anthropic's rivalry with OpenAI heats up with its claim new Claude AI surpasses GPT-4o",
 'body': 'Just a month after OpenAI rolled out its latest AI model, GPT-4o, today its rival Anthropic—famously founded by breakaway OpenAI researchers in 2021—said it had developed a new model to top it.',
 'url': 'https://www.msn.com/en-us/news/technology/anthropic-s-rivalry-with-openai-heats-up-with-its-claim-new-claude-ai-surpasses-gpt-4o/ar-BB1oApe1',
 'image': 'https://img-s-msn-com.akamaized.net/tenant/amp/entityid/BB1oAbpV.img?w=2048&h=1366&m=4&q=88',
 'source': 'Fortune on MSN.com',
 'found_at': datetime.datetime(2024, 6, 26, 9, 51, 56, 553000)}

In [2]:
# Type assertions

assert collection.count_documents({"date": {"$not": {"$type": "date"}}}) == 0
assert collection.count_documents({"found_at": {"$not": {"$type": "date"}}}) == 0

# Load CSV


In [12]:
import pandas as pd

CSV_PATH = r"data\blogdb.ai_news_2024_07_04_10_02_56.csv"

df = pd.read_csv(CSV_PATH, parse_dates=["date", "found_at"])

print(f"Number of rows: {len(df)}")

df.head()

Number of rows: 36266


,_id,date,title,body,url,image,source,found_at,page_content,domain,region
0,667d1ef6fc45eb48396a6a0c,2024-06-20 14:00:00+00:00,Anthropic's rivalry with OpenAI heats up with ...,Just a month after OpenAI rolled out its lates...,https://www.msn.com/en-us/news/technology/anth...,https://img-s-msn-com.akamaized.net/tenant/amp...,Fortune on MSN.com,2024-06-26 09:51:56.553000+00:00,NaN,msn.com,wt-wt
1,667d1ef6fc45eb48396a6a07,2024-06-25 12:00:00+00:00,Etched is building an AI chip that only runs o...,"The transformer, proposed by a team of Google ...",https://techcrunch.com/2024/06/25/etched-is-bu...,https://techcrunch.com/wp-content/uploads/2023...,TechCrunch,2024-06-26 09:51:56.553000+00:00,As generative AI touches a growing number of i...,techcrunch.com,wt-wt
2,667d1ef6fc45eb48396a6a06,2024-06-25 21:49:00+00:00,Boston scientists create AI model to 'catch Al...,Researchers say they've created a promising AI...,https://www.msn.com/en-us/health/other/boston-...,https://img-s-msn-com.akamaized.net/tenant/amp...,Tribune News Service on MSN.com,2024-06-26 09:51:56.553000+00:00,NaN,msn.com,wt-wt
3,667d1ef6fc45eb48396a6a0f,2024-06-25 18:07:00+00:00,FDA clears new AI-powered 12-lead ECG from Ali...,AliveCor announced today that it received FDA ...,https://www.massdevice.com/fda-clears-ai-12-le...,https://www.massdevice.com/wp-content/uploads/...,MassDevice,2024-06-26 09:51:56.553000+00:00,AliveCor announced today that it received FDA ...,massdevice.com,wt-wt
4,667d1ef6fc45eb48396a6a11,2024-06-26 08:00:00+00:00,EasyTranslate thinks augmenting LLMs with huma...,But now it's headed in a new direction with a ...,https://techcrunch.com/2024/06/26/easytranslat...,NaN,TechCrunch,2024-06-26 09:51:56.553000+00:00,You might think new generative AI startups lik...,techcrunch.com,wt-wt


In [14]:
# When duplicates are found based on URL, we should keep the one having an ID
# so we 1st sort by URL and then by ID in descending order. This way, the one with an ID will be kept.

df = df.sort_values(by=["url", "_id"], ascending=[True, False]).drop_duplicates(
    subset=["url"], keep="first"
)

print(f"Number of rows: {len(df)}")

Number of rows: 35941


# Insert data in Mongodb


In [10]:
# drop _id column and convert df in json format

data = df.drop(columns=["_id"]).to_dict(orient="records")

for x in data:
    if isinstance(x["date"], str):
        x["date"] = datetime.fromisoformat(x["date"])
    if isinstance(x["found_at"], str):
        x["found_at"] = datetime.fromisoformat(x["found_at"])

assert all(isinstance(x["date"], datetime) for x in data)
assert all(isinstance(x["found_at"], datetime) for x in data)

data[:2]

[{'date': Timestamp('2024-07-01 08:33:00+0000', tz='UTC'),
  'title': 'EDB Conducts an Awareness Seminar on Smart Solutions for Agriculture & Aquaculture Exporters',
  'body': 'The Sri Lanka Export Development Board (EDB) recently hosted an insightful seminar titled "Smart Solutions for Agricultural & Aquaculture Exporters", together with a smart solutions providing company,',
  'url': 'http://bizenglish.adaderana.lk/edb-conducts-an-awareness-seminar-on-smart-solutions-for-agriculture-aquaculture-exporters/',
  'image': 'http://s3.amazonaws.com/bizenglish/wp-content/uploads/2024/07/01140024/11-2-e1719822637574.jpg',
  'source': 'bizenglish.adaderana',
  'found_at': datetime.datetime(2024, 7, 2, 7, 41, 27, 231000, tzinfo=datetime.timezone.utc),
  'page_content': 'The Sri Lanka Export Development Board (EDB) recently hosted an insightful seminar titled "Smart Solutions for Agricultural & Aquaculture Exporters", together with a smart solutions providing company, E Gravity Solutions Pvt Lt

In [11]:
from pymongo.errors import BulkWriteError


try:
    result = collection.insert_many(data, ordered=False)
    print(f"Inserted {len(result.inserted_ids)} documents")
except BulkWriteError as e:
    print(f"Inserted {e.details['nInserted']} documents")
    print(f"Encountered {len(e.details['writeErrors'])} errors")  # number of duplicates

Inserted 9835 documents
Encountered 26106 errors
